# 01 — Supervised Fine-Tuning (SFT)

This notebook fine-tunes **Qwen/Qwen2.5-7B-Instruct** on the GSM8K dataset
using QLoRA + LoRA adapters via the `math_rl_tuning` package.

**Requirements:** Google Colab with GPU (T4 minimum, A100 recommended).

## 1. Setup — Install & Clone

In [1]:
# Clone the repo (replace with your GitHub URL)
!git clone https://github.com/Roogard/math-rl-tuning.git
%cd math-rl-tuning

# Install the package and all dependencies
!pip install -e . --quiet
!pip install bitsandbytes --quiet

Cloning into 'math-rl-tuning'...
remote: Enumerating objects: 411, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 411 (delta 30), reused 41 (delta 19), pack-reused 352 (from 1)
Receiving objects: 100% (411/411), 4.17 MiB | 4.33 MiB/s, done.
Resolving deltas: 100% (254/254), done.
/content/math-rl-tuning
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 12.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.8/89.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 53.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4

## 2. Configuration

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA alloc conf: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF', 'default')}")

In [3]:
from math_rl_tuning.config import load_config
from math_rl_tuning.utils import setup_hf_token, setup_wandb, mount_google_drive

# Load default config (edit configs/default.yaml to customize)
cfg = load_config()

# --- Authentication ---
# Option A: Set your tokens here
# setup_hf_token("hf_YOUR_TOKEN")
# setup_wandb(cfg.sft_training.wandb_project, key="YOUR_WANDB_KEY")

# Option B: Use Colab secrets (recommended)
setup_hf_token()  # reads from Colab secrets
setup_wandb(cfg.sft_training.wandb_project)

# Mount Google Drive for saving
mount_google_drive()

# --- Checkpoint Resume ---
# To resume from a crashed run, paste the checkpoint folder path here.
# Checkpoints are saved every 100 steps to outputs/sft/checkpoint-*/
# Leave as None to start fresh.
SFT_CHECKPOINT = None  # e.g. "/content/drive/MyDrive/math-rl-tuning/sft/checkpoint-100"

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Mounted at /content/drive


## 3. (Optional) Customize Config

You can override any config value programmatically:

In [4]:
# Example: train for 2 epochs with a larger batch size
# cfg.sft_training.num_train_epochs = 2
# cfg.sft_training.per_device_train_batch_size = 8

# Example: use different data sources
# cfg.dataset.sft_keep_sources = ["gsm8k", "math", "cn_k12"]

# Example: change LoRA rank
# cfg.lora.sft.r = 32
# cfg.lora.sft.alpha = 64

## 4. Prepare Data

In [5]:
from math_rl_tuning.data import prepare_sft_data

train_ds, val_ds = prepare_sft_data(cfg)

print(f"\nTrain examples: {len(train_ds)}")
print(f"Val examples:   {len(val_ds)}")
print(f"\nSample (first message):")
print(train_ds[0]["messages"][0]["content"][:500])

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

...

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Normalizing answer format to \boxed{}...


Map:   0%|          | 0/859494 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Sources kept: ['gsm8k', 'math']
Building balanced splits...
  gsm8k: train=6000, val=1000
  math: train=6000, val=1000
Train: 12000  |  Val: 2000

Train examples: 12000
Val examples:   2000

Sample (first message):
Please reason step by step, and put your final answer within \boxed{}.


## 5. Run SFT Training

In [6]:
from math_rl_tuning.sft_trainer import run_sft_training

trainer, model, tokenizer = run_sft_training(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    save_to_drive=True,  # auto-copies to Google Drive
    checkpoint_path=SFT_CHECKPOINT,
)


PHASE 2: Model Loading
Loading tokenizer: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Step,Training Loss
25,0.499221
50,0.307470
75,0.285640
100,0.284018
125,0.262313
150,0.277491
175,0.268037
200,0.261622
225,0.275113
250,0.273082


wandb: WARNING URL not available in offline run
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
wandb: WARNING URL not available in offline run
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
wandb: WARNING URL not available in offline run
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
wandb: WARNING URL not available in offline run
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warning


PHASE 4: Saving


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Model saved to: ./outputs/sft
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied to Drive: /content/drive/MyDrive/math-rl-tuning/sft


## 6. Quick Sanity Check

In [7]:
from math_rl_tuning.inference import generate_stream

question = "Solve x + y = 10, 2x - y = 30."
print(f"Question: {question}\n")
response = generate_stream(question, model, tokenizer)

Question: Solve x + y = 10, 2x - y = 30.

To solve the system of equations given by $x + y = 10$ and $2x - y = 30$, we can use the method of substitution or elimination. Here, we'll use a combination of both for clarity.

First, let's add the two equations together to eliminate $y$:

\[
\begin{align*}
(x + y) + (2x - y) &= 10 + 30 \\
x + y + 2x - y &= 40 \\
3x &= 40 \\
x &= \frac{40}{3} \\
x &= 13.\overline{3}
\end{align*}
\]

Now that we have $x = 13.\overline{3}$, we substitute this value back into one of the original equations to find $y$. Let's use the first equation $x + y = 10$:

\[
\begin{align*}
13.\overline{3} + y &= 10 \\
y &= 10 - 13.\overline{3} \\
y &= -3.\overline{3}
\end{align*}
\]

Therefore, the solution to the system of equations is $\boxed{(13.\overline{3}, -3.\overline{3})}$.


## 7. Cleanup

In [8]:
from math_rl_tuning.utils import clean_memory

del model, trainer
clean_memory()

Memory cleared.
